In [23]:
%pip install requests


Note: you may need to restart the kernel to use updated packages.


In [24]:
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed

API_URL = "http://127.0.0.1:8000"

def transfer(i):
    data = {
        "transfer_id": f"ISOLATION-TEST-{i:03d}",
        "from_account": "TEST-ACC-001",
        "to_account": "TEST-ACC-002",
        "amount": 10,
        "description": "Lab 10.3 concurrent isolation test"
    }

    try:
        response = requests.post(
            f"{API_URL}/bank/transfer",
            json=data,
            timeout=30
        )

        return {
            "transfer_id": data["transfer_id"],
            "status_code": response.status_code,
            "response": response.json()
        }

    except Exception as e:
        return {
            "transfer_id": data["transfer_id"],
            "error": str(e)
        }


results = []

print("Starting 8 concurrent bank transfers...\n")

with ThreadPoolExecutor(max_workers=8) as executor:
    futures = [executor.submit(transfer, i) for i in range(1, 9)]

    for future in as_completed(futures):
        result = future.result()
        results.append(result)
        print(result)

successful = sum(
    1 for result in results
    if result.get("status_code") == 200
)

failed = len(results) - successful

print("\n========== RESULTS ==========")
print("Total requests:", len(results))
print("Successful transfers:", successful)
print("Failed transfers:", failed)

Starting 8 concurrent bank transfers...

{'transfer_id': 'ISOLATION-TEST-003', 'error': "('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))"}
{'transfer_id': 'ISOLATION-TEST-001', 'error': "('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))"}
{'transfer_id': 'ISOLATION-TEST-008', 'error': "('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))"}
{'transfer_id': 'ISOLATION-TEST-004', 'error': "('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))"}
{'transfer_id': 'ISOLATION-TEST-002', 'error': "('Connection aborted.', ConnectionAbortedError(10053, 'An established connection was aborted by the software in your host machine', None, 10053, None))"}
{'transfer_id': 'ISOLATION-TEST-005', 'error': "('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))"}
{'transfer_id': 'ISOLATION-TEST-007', 'err

Transaction sessions are created inside the API's `/bank/transfer` endpoint and are not available in this notebook. Use the concurrent transfer test above to verify transaction isolation.

In [25]:
session.start_transaction()


NameError: name 'session' is not defined

In [ ]:
%pip install pymongo


Note: you may need to restart the kernel to use updated packages.


In [ ]:
from pymongo import MongoClient, WriteConcern
from datetime import datetime, timezone

MONGO_URI = "mongodb://localhost:27217/?replicaSet=rs0&directConnection=true"

client = MongoClient(
    MONGO_URI,
    serverSelectionTimeoutMS=8000,
    connectTimeoutMS=5000
)

print(client.admin.command("hello"))

{'topologyVersion': {'processId': ObjectId('6ae50216944ae1214bc834be'), 'counter': 18}, 'hosts': ['mongo1:27017', 'mongo2:27017'], 'passives': ['mongo3:27017'], 'setName': 'rs0', 'setVersion': 1, 'isWritablePrimary': True, 'secondary': False, 'primary': 'mongo1:27017', 'me': 'mongo1:27017', 'electionId': ObjectId('7fffffff0000000000000008'), 'lastWrite': {'opTime': {'ts': Timestamp(1793441483, 1), 't': 8}, 'lastWriteDate': datetime.datetime(2026, 10, 31, 10, 11, 23), 'majorityOpTime': {'ts': Timestamp(1793441483, 1), 't': 8}, 'majorityWriteDate': datetime.datetime(2026, 10, 31, 10, 11, 23)}, 'maxBsonObjectSize': 16777216, 'maxMessageSizeBytes': 48000000, 'maxWriteBatchSize': 100000, 'localTime': datetime.datetime(2026, 10, 31, 10, 11, 28, 958000), 'logicalSessionTimeoutMinutes': 30, 'connectionId': 17309, 'minWireVersion': 0, 'maxWireVersion': 21, 'readOnly': False, 'ok': 1.0, '$clusterTime': {'clusterTime': Timestamp(1793441483, 1), 'signature': {'hash': b'\x00\x00\x00\x00\x00\x00\x00

In [ ]:
db = client["lab10_app"]

durability = db.get_collection(
    "durability_test",
    write_concern=WriteConcern(w="majority", j=True)
)

print("Write concern:", durability.write_concern.document)

Write concern: {'j': True, 'w': 'majority'}


In [ ]:
test_document = {
    "test_id": "DURABILITY-TEST-001",
    "message": "Lab 10.3 durability verification",
    "status": "committed",
    "created_at": datetime.now(timezone.utc)
}

result = durability.insert_one(test_document)

print("Inserted ID:", result.inserted_id)
print("Write acknowledged:", result.acknowledged)

Inserted ID: 6ae5bedc33a5f48528acedac
Write acknowledged: True


In [ ]:
saved = durability.find_one(
    {"test_id": "DURABILITY-TEST-001"},
    {"_id": 0}
)

print("Saved record:", saved)

Saved record: {'test_id': 'DURABILITY-TEST-001', 'message': 'Lab 10.3 durability verification', 'status': 'committed', 'created_at': datetime.datetime(2026, 10, 31, 10, 11, 40, 758000)}


In [ ]:
from pymongo import MongoClient

ports = [27217, 27218, 27219]

for port in ports:
    try:
        node = MongoClient(
            f"mongodb://localhost:{port}/?directConnection=true",
            serverSelectionTimeoutMS=3000
        )

        record = node["lab10_app"]["durability_test"].find_one(
            {"test_id": "DURABILITY-TEST-001"},
            {"_id": 0}
        )

        print(f"\nPort: {port}")
        print("Record found:", record is not None)
        print("Record:", record)

        node.close()

    except Exception as e:
        print(f"\nPort {port}: Connection failed - {e}")


Port: 27217
Record found: True
Record: {'test_id': 'DURABILITY-TEST-001', 'message': 'Lab 10.3 durability verification', 'status': 'committed', 'created_at': datetime.datetime(2026, 10, 31, 10, 11, 40, 758000)}

Port: 27218
Record found: True
Record: {'test_id': 'DURABILITY-TEST-001', 'message': 'Lab 10.3 durability verification', 'status': 'committed', 'created_at': datetime.datetime(2026, 10, 31, 10, 11, 40, 758000)}

Port: 27219
Record found: True
Record: {'test_id': 'DURABILITY-TEST-001', 'message': 'Lab 10.3 durability verification', 'status': 'committed', 'created_at': datetime.datetime(2026, 10, 31, 10, 11, 40, 758000)}


In [ ]:
from pymongo import MongoClient

client = MongoClient(
    "mongodb://localhost:27217,localhost:27218,localhost:27219/?replicaSet=rs0",
    serverSelectionTimeoutMS=8000
)

status = client.admin.command("replSetGetStatus")

for member in status["members"]:
    print(
        "Member:", member["name"],
        "| State:", member["stateStr"],
        "| Health:", member["health"]
    )

ServerSelectionTimeoutError: mongo2:27017: [Errno 11001] getaddrinfo failed (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms),mongo3:27017: [Errno 11001] getaddrinfo failed (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms),mongo1:27017: [Errno 11001] getaddrinfo failed (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms), Timeout: 8.0s, Topology Description: <TopologyDescription id: 6ae5bf2c33a5f48528acedb0, topology_type: ReplicaSetNoPrimary, servers: [<ServerDescription ('mongo1', 27017) server_type: Unknown, rtt: None, error=AutoReconnect('mongo1:27017: [Errno 11001] getaddrinfo failed (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms)')>, <ServerDescription ('mongo2', 27017) server_type: Unknown, rtt: None, error=AutoReconnect('mongo2:27017: [Errno 11001] getaddrinfo failed (configured timeouts: socketTimeoutMS: 20000.0ms, connectTimeoutMS: 20000.0ms)')>, <ServerDescripti

In [ ]:
from pymongo import MongoClient

client = MongoClient(
    "mongodb://localhost:27217/?directConnection=true",
    serverSelectionTimeoutMS=8000
)

status = client.admin.command("replSetGetStatus")

for member in status["members"]:
    print(
        "Member:", member["name"],
        "| State:", member["stateStr"],
        "| Health:", member["health"]
    )
    

Member: mongo1:27017 | State: PRIMARY | Health: 1.0
Member: mongo2:27017 | State: SECONDARY | Health: 1.0
Member: mongo3:27017 | State: SECONDARY | Health: 1.0


In [ ]:
from pymongo import MongoClient
from pymongo.errors import PyMongoError

client = MongoClient(
    "mongodb://localhost:27217/?directConnection=true",
    serverSelectionTimeoutMS=8000
)

try:
    result = client.admin.command({
        "replSetStepDown": 60
    })
    print(result)
except PyMongoError as e:
    print("Primary stepped down or connection was interrupted:")
    print(e)

{'ok': 1.0, '$clusterTime': {'clusterTime': Timestamp(1793441733, 1), 'signature': {'hash': b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00', 'keyId': 0}}, 'operationTime': Timestamp(1793441733, 1)}


In [ ]:
from pymongo import MongoClient

client = MongoClient(
    "mongodb://localhost:27218/?directConnection=true",
    serverSelectionTimeoutMS=15000
)

status = client.admin.command("replSetGetStatus")

for member in status["members"]:
    print(
        "Member:", member["name"],
        "| State:", member["stateStr"],
        "| Health:", member["health"]
    )

Member: mongo1:27017 | State: SECONDARY | Health: 1.0
Member: mongo2:27017 | State: PRIMARY | Health: 1.0
Member: mongo3:27017 | State: SECONDARY | Health: 1.0


In [ ]:
from pymongo import MongoClient

client = MongoClient(
    "mongodb://localhost:27218/?directConnection=true",
    serverSelectionTimeoutMS=15000
)

db = client["lab10_app"]

record = db["durability_test"].find_one({
    "test_id": "DURABILITY-TEST-001"
})

if record:
    print("PASS: Committed record survived primary failover!")
    print(record)
else:
    print("FAIL: Record not found.")
    

PASS: Committed record survived primary failover!
{'_id': ObjectId('6ae5bedc33a5f48528acedac'), 'test_id': 'DURABILITY-TEST-001', 'message': 'Lab 10.3 durability verification', 'status': 'committed', 'created_at': datetime.datetime(2026, 10, 31, 10, 11, 40, 758000)}


In [ ]:
from pymongo import MongoClient

client = MongoClient(
    "mongodb://localhost:27218/?directConnection=true",
    serverSelectionTimeoutMS=15000
)

print("Databases:", client.list_database_names())

db = client["cti_intel"]

print("Collections:", db.list_collection_names())

if "threats" in db.list_collection_names():
    print("Total threats:", db["threats"].count_documents({}))
    print("\nSample threat:")
    print(db["threats"].find_one())
else:
    print("The threats collection is not available in this database.")

Databases: ['admin', 'config', 'lab10_app', 'local']
Collections: []
The threats collection is not available in this database.


In [ ]:
from pymongo import MongoClient

client7 = MongoClient(
    "mongodb://localhost:27117/?directConnection=true",
    serverSelectionTimeoutMS=10000
)

print("Lab 7 databases:", client7.list_database_names())

db7 = client7["cti_intel"]

print("Collections:", db7.list_collection_names())

if "threats" in db7.list_collection_names():
    print("Total threats:", db7["threats"].count_documents({}))
    print("\nSample threat:")
    print(db7["threats"].find_one())

Lab 7 databases: ['admin', 'config', 'cti_cisa_kev', 'cti_digest', 'cti_feodo_tracker', 'cti_malwarebazaar', 'cti_openphish', 'cti_urlhaus', 'cyber_threat_intelligence', 'failover_test', 'lab10_app', 'lab_9_1_try_yourself', 'lab_9_1_write_concern', 'lab_9_2', 'local', 'testdb']
Collections: []


In [ ]:
from pymongo import MongoClient

client7 = MongoClient(
    "mongodb://localhost:27117/?directConnection=true",
    serverSelectionTimeoutMS=10000
)

for db_name in client7.list_database_names():
    if db_name in ["admin", "config", "local"]:
        continue

    db = client7[db_name]

    for collection_name in db.list_collection_names():
        collection = db[collection_name]
        count = collection.count_documents({})

        print(
            f"Database: {db_name} | "
            f"Collection: {collection_name} | "
            f"Documents: {count}"
        )

Database: cti_cisa_kev | Collection: sync_log | Documents: 7
Database: cti_cisa_kev | Collection: indicators | Documents: 1766
Database: cti_cisa_kev | Collection: cti_iocs | Documents: 1766
Database: cti_cisa_kev | Collection: source_metadata | Documents: 1
Database: cti_digest | Collection: cti_iocs | Documents: 1
Database: cti_digest | Collection: intelligence | Documents: 1
Database: cti_digest | Collection: sync_log | Documents: 7
Database: cti_digest | Collection: indicators | Documents: 1
Database: cti_digest | Collection: cti_articles | Documents: 1
Database: cti_digest | Collection: source_metadata | Documents: 1
Database: cti_feodo_tracker | Collection: cti_iocs | Documents: 5
Database: cti_feodo_tracker | Collection: indicators | Documents: 5
Database: cti_feodo_tracker | Collection: sync_log | Documents: 7
Database: cti_feodo_tracker | Collection: source_metadata | Documents: 1
Database: cti_malwarebazaar | Collection: cti_iocs | Documents: 0
Database: cti_malwarebazaar | C

In [ ]:
from pymongo import MongoClient
from pprint import pprint

client7 = MongoClient(
    "mongodb://localhost:27117/?directConnection=true"
)

collection = client7["cyber_threat_intelligence"]["working_set"]

print("Total documents:", collection.count_documents({}))

print("\nSample threat record:")
pprint(collection.find_one())

Total documents: 100000

Sample threat record:
{'_id': ObjectId('6ae4ca25c1093791dcd79d7f'),
 'created_at': datetime.datetime(2026, 10, 30, 16, 47, 33, 796000),
 'key': 'sensor-000000',
 'payload': 'cti-working-set',
 'seq': 0,
 'value': 0.8527407685113338}


In [ ]:
from pymongo import MongoClient
from pprint import pprint

client7 = MongoClient(
    "mongodb://localhost:27117/?directConnection=true"
)

collections = [
    ("cti_cisa_kev", "indicators"),
    ("cti_feodo_tracker", "cti_iocs"),
    ("cti_openphish", "indicators")
]

for db_name, collection_name in collections:
    print("\n" + "=" * 50)
    print("Database:", db_name)
    print("Collection:", collection_name)

    collection = client7[db_name][collection_name]
    print("Documents:", collection.count_documents({}))

    sample = collection.find_one()
    pprint(sample)


Database: cti_cisa_kev
Collection: indicators
Documents: 1766
{'_id': ObjectId('6ae4c1606a381ec89225ab52'),
 'first_seen': datetime.datetime(2026, 10, 30, 16, 10, 8, 366000),
 'indicator': 'CVE-2026-97229',
 'indicator_type': 'cve',
 'ioc_key': '0b2caa9fae60a4abd6a44f96fd402a4af0670f67c4b0536658eef546bc8d0262',
 'ip': None,
 'mention_count': 1,
 'source': 'CISA KEV',
 'updated_at': datetime.datetime(2026, 10, 31, 7, 45, 15, 404000)}

Database: cti_feodo_tracker
Collection: cti_iocs
Documents: 5
{'_id': ObjectId('6ae4c1636a381ec89225d4b2'),
 'category': 'botnet-c2',
 'confidence': 'public community feed',
 'context': '{"port": 8080, "status": "offline", "hostname": null, '
            '"as_number": 14061, "as_name": "DIGITALOCEAN-ASN", "country": '
            '"US", "last_online": "2026-03-07", "malware": "Emotet"}',
 'created_at': datetime.datetime(2026, 10, 30, 16, 10, 11, 49000),
 'first_seen': '2022-06-04 21:24:53',
 'indicator': '162.243.103.246',
 'indicator_type': 'ip',
 'ioc_k

In [ ]:
from pymongo import MongoClient
from pprint import pprint

client7 = MongoClient(
    "mongodb://localhost:27117/?directConnection=true"
)

threats = client7["cti_cisa_kev"]["indicators"]

threat = threats.find_one({
    "indicator": "CVE-2026-97229"
})

if threat:
    print("Threat record found:")
    pprint(threat)
else:
    print("Threat record not found.")

Threat record found:
{'_id': ObjectId('6ae4c1606a381ec89225ab52'),
 'first_seen': datetime.datetime(2026, 10, 30, 16, 10, 8, 366000),
 'indicator': 'CVE-2026-97229',
 'indicator_type': 'cve',
 'ioc_key': '0b2caa9fae60a4abd6a44f96fd402a4af0670f67c4b0536658eef546bc8d0262',
 'ip': None,
 'mention_count': 1,
 'source': 'CISA KEV',
 'updated_at': datetime.datetime(2026, 10, 31, 7, 45, 15, 404000)}


In [ ]:
from pymongo import MongoClient

# Original Lab 7 cluster — read only
lab7 = MongoClient(
    "mongodb://localhost:27117/?directConnection=true"
)

source = lab7["cti_cisa_kev"]["indicators"]

threat = source.find_one({
    "indicator": "CVE-2026-97229"
})

if threat is None:
    print("Threat not found in Lab 7.")
else:
    # Lab 10 cluster — create a working copy
    lab10 = MongoClient(
        "mongodb://localhost:27218/?directConnection=true"
    )

    target = lab10["cti_lab10"]["threats"]

    threat_copy = dict(threat)
    threat_copy["blocked_flag"] = False
    threat_copy["source_database"] = "cti_cisa_kev"
    threat_copy["source_collection"] = "indicators"

    target.replace_one(
        {"_id": threat_copy["_id"]},
        threat_copy,
        upsert=True
    )

    print("Working copy created successfully!")
    print(target.find_one({
        "indicator": "CVE-2026-97229"
    }))

NotPrimaryError: not primary, full error: {'topologyVersion': {'processId': ObjectId('6ae50216081168f77da967bd'), 'counter': 27}, 'ok': 0.0, 'errmsg': 'not primary', 'code': 10107, 'codeName': 'NotWritablePrimary', '$clusterTime': {'clusterTime': Timestamp(1793442004, 1), 'signature': {'hash': b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00', 'keyId': 0}}, 'operationTime': Timestamp(1793442004, 1)}

In [ ]:
from pymongo import MongoClient

for port in [27217, 27218, 27219]:
    try:
        client = MongoClient(
            f"mongodb://localhost:{port}/?directConnection=true",
            serverSelectionTimeoutMS=5000
        )

        result = client.admin.command("hello")

        print(
            f"Port {port} | "
            f"Primary: {result.get('isWritablePrimary')} | "
            f"Member: {result.get('me')}"
        )

        client.close()

    except Exception as e:
        print(f"Port {port} | Error: {e}")

Port 27217 | Primary: True | Member: mongo1:27017
Port 27218 | Primary: False | Member: mongo2:27017
Port 27219 | Primary: False | Member: mongo3:27017


In [ ]:
from pymongo import MongoClient

# Read the original Lab 7 record
lab7 = MongoClient(
    "mongodb://localhost:27117/?directConnection=true"
)

threat = lab7["cti_cisa_kev"]["indicators"].find_one({
    "indicator": "CVE-2026-97229"
})

if threat is None:
    print("Threat not found!")
else:
    # Connect to the current Lab 10 PRIMARY
    lab10 = MongoClient(
        "mongodb://localhost:27217/?directConnection=true"
    )

    target = lab10["cti_lab10"]["threats"]

    threat_copy = dict(threat)
    threat_copy["blocked_flag"] = False
    threat_copy["source_database"] = "cti_cisa_kev"
    threat_copy["source_collection"] = "indicators"

    target.replace_one(
        {"_id": threat_copy["_id"]},
        threat_copy,
        upsert=True
    )

    print("Working copy created successfully!")
    print(target.find_one({
        "indicator": "CVE-2026-97229"
    }))

Working copy created successfully!
{'_id': ObjectId('6ae4c1606a381ec89225ab52'), 'ioc_key': '0b2caa9fae60a4abd6a44f96fd402a4af0670f67c4b0536658eef546bc8d0262', 'first_seen': datetime.datetime(2026, 10, 30, 16, 10, 8, 366000), 'indicator': 'CVE-2026-97229', 'indicator_type': 'cve', 'ip': None, 'mention_count': 1, 'source': 'CISA KEV', 'updated_at': datetime.datetime(2026, 10, 31, 7, 45, 15, 404000), 'blocked_flag': False, 'source_database': 'cti_cisa_kev', 'source_collection': 'indicators'}


In [26]:
from pymongo import MongoClient
from pymongo.errors import PyMongoError
from datetime import datetime, timezone
from pprint import pprint

client = MongoClient(
    "mongodb://localhost:27217/?directConnection=true",
    serverSelectionTimeoutMS=15000
)

db = client["cti_lab10"]

threats = db["threats"]
blocklist = db["blocklist"]
audit_log = db["audit_log"]

indicator = "CVE-2026-97229"

def block_threat(fail_midway=False):
    with client.start_session() as session:

        def transaction_work(session):
            threat = threats.find_one(
                {"indicator": indicator},
                session=session
            )

            if threat is None:
                raise ValueError("Threat not found")

            if threat.get("blocked_flag", False):
                raise ValueError("Threat is already blocked")

            threats.update_one(
                {"indicator": indicator},
                {"$set": {"blocked_flag": True}},
                session=session
            )

            blocklist.insert_one({
                "indicator": indicator,
                "indicator_type": threat["indicator_type"],
                "source": threat["source"],
                "blocked_at": datetime.now(timezone.utc)
            }, session=session)

            if fail_midway:
                raise RuntimeError("Simulated failure")

            audit_log.insert_one({
                "indicator": indicator,
                "action": "THREAT_BLOCKED",
                "timestamp": datetime.now(timezone.utc)
            }, session=session)

            return "Threat blocked successfully"

        return session.with_transaction(transaction_work)

print("Transaction function created successfully.")

Transaction function created successfully.


In [27]:
# Simulate failure in the middle of the transaction
try:
    block_threat(fail_midway=True)
except RuntimeError as e:
    print("Expected failure:", e)

# Verify all three collections
threat = threats.find_one({"indicator": indicator})

blocklist_count = blocklist.count_documents({
    "indicator": indicator
})

audit_count = audit_log.count_documents({
    "indicator": indicator
})

print("\n--- ATOMICITY VERIFICATION ---")
print("Threat blocked_flag:", threat["blocked_flag"])
print("Blocklist records:", blocklist_count)
print("Audit records:", audit_count)

if (
    threat["blocked_flag"] is False
    and blocklist_count == 0
    and audit_count == 0
):
    print("\nPASS: Transaction rolled back completely!")
else:
    print("\nCHECK: Unexpected data found.")

Expected failure: Simulated failure

--- ATOMICITY VERIFICATION ---
Threat blocked_flag: False
Blocklist records: 0
Audit records: 0

PASS: Transaction rolled back completely!


In [28]:
# Execute the transaction successfully
try:
    result = block_threat(fail_midway=False)
    print("Transaction result:", result)
except Exception as e:
    print("Transaction failed:", e)

# Verify all three collections
threat = threats.find_one({"indicator": indicator})

blocklist_count = blocklist.count_documents({
    "indicator": indicator
})

audit_count = audit_log.count_documents({
    "indicator": indicator
})

print("\n--- SUCCESSFUL TRANSACTION VERIFICATION ---")
print("Threat blocked_flag:", threat["blocked_flag"])
print("Blocklist records:", blocklist_count)
print("Audit records:", audit_count)

if (
    threat["blocked_flag"] is True
    and blocklist_count == 1
    and audit_count == 1
):
    print("\nPASS: All three operations committed successfully!")
else:
    print("\nCHECK: Transaction results need investigation.")

Transaction result: Threat blocked successfully

--- SUCCESSFUL TRANSACTION VERIFICATION ---
Threat blocked_flag: True
Blocklist records: 1
Audit records: 1

PASS: All three operations committed successfully!


In [29]:
# Attempt to block the same threat again
try:
    result = block_threat(fail_midway=False)
    print("Unexpected result:", result)

except ValueError as e:
    print("Expected rejection:", e)

# Verify that no duplicate records were created
blocklist_count = blocklist.count_documents({
    "indicator": indicator
})

audit_count = audit_log.count_documents({
    "indicator": indicator
})

print("\n--- DUPLICATE PREVENTION VERIFICATION ---")
print("Blocklist records:", blocklist_count)
print("Audit records:", audit_count)

if blocklist_count == 1 and audit_count == 1:
    print("\nPASS: Duplicate blocking prevented!")
else:
    print("\nCHECK: Duplicate records detected.")

Expected rejection: Threat is already blocked

--- DUPLICATE PREVENTION VERIFICATION ---
Blocklist records: 1
Audit records: 1

PASS: Duplicate blocking prevented!


In [30]:
from pymongo import MongoClient

client = MongoClient(
    "mongodb://localhost:27217/?directConnection=true",
    serverSelectionTimeoutMS=10000
)

status = client.admin.command("replSetGetStatus")

for member in status["members"]:
    print(
        member["name"],
        "| State:", member["stateStr"],
        "| Health:", member["health"]
    )

mongo1:27017 | State: PRIMARY | Health: 1.0
mongo2:27017 | State: SECONDARY | Health: 1.0
mongo3:27017 | State: SECONDARY | Health: 1.0


In [31]:
from pymongo import MongoClient

client = MongoClient(
    "mongodb://localhost:27217/?directConnection=true",
    serverSelectionTimeoutMS=10000
)

status = client.admin.command("replSetGetStatus")

for member in status["members"]:
    print(
        member["name"],
        "| State:", member["stateStr"],
        "| Health:", member["health"]
    )

mongo1:27017 | State: PRIMARY | Health: 1.0
mongo2:27017 | State: SECONDARY | Health: 1.0
mongo3:27017 | State: (not reachable/healthy) | Health: 0.0


In [32]:
docker compose exec api python -c "import app; print('MONGO_URI:', app.MONGO_URI); print('Client:', app.client.address)"


SyntaxError: Missing parentheses in call to 'exec'. Did you mean exec(...)? (2316278909.py, line 1)